# Yoga Pose Classification — EfficientNetB0 + EfficientNetB2 Stacking Ensemble

**Full methodology:**
- GPU memory growth
- Rich data augmentation (rotation, shift, zoom, flip, brightness, channel-shift, MixUp)
- L1/L2 regularisation + Dropout + BatchNorm
- Label smoothing + Gradient clipping (SGD + clipnorm)
- Stratified K-Fold (k=3) two-phase training (freeze → fine-tune)
- Class-weight balancing for imbalanced dataset
- Serialisation-safe saving: weights-only `.weights.h5`
- SVM meta-learner stacking ensemble
- Full evaluation: accuracy, classification report, confusion matrix


In [1]:
import os, glob, gc, time, json, warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications import EfficientNetB0, EfficientNetB2
from tensorflow.keras.layers import (
    Dense, GlobalAveragePooling2D, Dropout, BatchNormalization, Input
)
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import SGD
from tensorflow.keras import regularizers
from tensorflow.keras.losses import CategoricalCrossentropy
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau

from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.svm import SVC
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
import joblib

# ── GPU ──────────────────────────────────────────────────────────────────────
gpus = tf.config.list_physical_devices('GPU')
if gpus:
    for gpu in gpus:
        tf.config.experimental.set_memory_growth(gpu, True)
    print(f'GPU Memory Growth enabled on {len(gpus)} GPU(s).')
else:
    print('No GPU found – running on CPU.')

print('TensorFlow:', tf.__version__)
tf.random.set_seed(42)
np.random.seed(42)


Could not save font_manager cache [Errno 13] Permission denied: 'C:\\Users\\ABHIROOP\\.matplotlib\\fontlist-v390.json.matplotlib-lock'


GPU Memory Growth enabled on 1 GPU(s).
TensorFlow: 2.10.0


In [2]:
DATASET_DIR = r'd:\Yoga Website\Dataset'
IMG_SIZE    = (224, 224)
BATCH_SIZE  = 16
K_FOLDS     = 3
EPOCHS      = 80    # EarlyStopping will cut this short
L1_RATE     = 1e-5
L2_RATE     = 5e-4
DROPOUT     = 0.5
LABEL_SMOOTH= 0.05
MIXUP_ALPHA = 0.2
os.makedirs('saved_models', exist_ok=True)


In [3]:
classes = sorted([d for d in os.listdir(DATASET_DIR)
                   if os.path.isdir(os.path.join(DATASET_DIR, d))])
NUM_CLASSES = len(classes)

filepaths, labels = [], []
for c in classes:
    for ext in ('*.jpg','*.jpeg','*.png','*.JPG','*.JPEG','*.PNG'):
        for p in glob.glob(os.path.join(DATASET_DIR, c, ext)):
            filepaths.append(p)
            labels.append(c)

df = pd.DataFrame({'filepath': filepaths, 'label': labels})

# Save class list for inference
with open('saved_models/class_names.json', 'w') as f:
    json.dump(classes, f)

# Holdout 10 % for final ensemble test
train_val_df, test_df = train_test_split(
    df, test_size=0.10, stratify=df['label'], random_state=42
)

print(f'Total images  : {len(df)}')
print(f'Classes       : {NUM_CLASSES}')
print(f'Train/Val     : {len(train_val_df)}')
print(f'Holdout Test  : {len(test_df)}')
print('Min images/class:', df.groupby('label').size().min())
print('Max images/class:', df.groupby('label').size().max())


Total images  : 5512
Classes       : 47
Train/Val     : 4960
Holdout Test  : 552
Min images/class: 24
Max images/class: 192


In [4]:
from sklearn.utils.class_weight import compute_class_weight

y_labels = train_val_df['label'].map({c: i for i, c in enumerate(classes)}).values
cw = compute_class_weight('balanced', classes=np.unique(y_labels), y=y_labels)
class_weight_dict = {i: w for i, w in enumerate(cw)}
print('Class weights computed (sample):', dict(list(class_weight_dict.items())[:5]))


Class weights computed (sample): {0: 0.7934730443129099, 1: 0.9019821785779233, 2: 3.297872340425532, 3: 0.824468085106383, 4: 0.9955841027699719}


In [5]:
train_datagen = ImageDataGenerator(
    rotation_range=30,
    width_shift_range=0.2,
    height_shift_range=0.2,
    zoom_range=0.3,
    horizontal_flip=True,
    fill_mode='nearest',
    brightness_range=[0.7, 1.3],
    channel_shift_range=20.0,
    # rescale handled by EfficientNet internally
)

val_datagen = ImageDataGenerator()


def mixup(x, y, alpha=MIXUP_ALPHA):
    lam = np.random.beta(alpha, alpha)
    idx = np.random.permutation(len(x))
    return lam * x + (1 - lam) * x[idx], lam * y + (1 - lam) * y[idx]


def mixup_generator(gen):
    """Infinite MixUp wrapper around a Keras generator."""
    while True:
        x, y = next(gen)
        yield mixup(x, y)


In [6]:
def build_model(base_fn):
    """
    base_fn : EfficientNetB0 or EfficientNetB2
    Returns  : (model, base_model)
    """
    inputs = Input(shape=(*IMG_SIZE, 3))
    base   = base_fn(weights='imagenet', include_top=False,
                     input_tensor=inputs)
    base.trainable = False

    x = base(inputs, training=False)
    x = GlobalAveragePooling2D()(x)

    x = Dense(512, activation='relu',
               kernel_regularizer=regularizers.l1_l2(l1=L1_RATE, l2=L2_RATE))(x)
    x = BatchNormalization()(x)
    x = Dropout(DROPOUT)(x)

    x = Dense(256, activation='relu',
               kernel_regularizer=regularizers.l1_l2(l1=L1_RATE, l2=L2_RATE))(x)
    x = BatchNormalization()(x)
    x = Dropout(DROPOUT)(x)

    outputs = Dense(NUM_CLASSES, activation='softmax')(x)
    model   = Model(inputs, outputs)
    return model, base


In [7]:
loss_fn = CategoricalCrossentropy(label_smoothing=LABEL_SMOOTH)


def get_callbacks():
    return [
        EarlyStopping(monitor='val_loss', patience=10,
                      restore_best_weights=True, verbose=1),
        ReduceLROnPlateau(monitor='val_loss', factor=0.5,
                          patience=5, min_lr=1e-7, verbose=1),
    ]


def train_kfold(base_fn, model_tag):
    """
    Stratified K-Fold training with two-phase strategy.
    Saves weights of best fold to  saved_models/{model_tag}_best.weights.h5
    Returns best_acc.
    """
    skf      = StratifiedKFold(n_splits=K_FOLDS, shuffle=True, random_state=42)
    best_acc = 0.0

    for fold, (tr_idx, vl_idx) in enumerate(
            skf.split(train_val_df['filepath'], train_val_df['label'])):

        print(f'\n{"="*50}')
        print(f'  {model_tag}  |  Fold {fold+1}/{K_FOLDS}')
        print(f'{"="*50}')

        fold_train = train_val_df.iloc[tr_idx]
        fold_val   = train_val_df.iloc[vl_idx]

        train_gen = train_datagen.flow_from_dataframe(
            fold_train, x_col='filepath', y_col='label',
            target_size=IMG_SIZE, batch_size=BATCH_SIZE,
            class_mode='categorical', shuffle=True)

        val_gen = val_datagen.flow_from_dataframe(
            fold_val, x_col='filepath', y_col='label',
            target_size=IMG_SIZE, batch_size=BATCH_SIZE,
            class_mode='categorical', shuffle=False)

        model, base = build_model(base_fn)

        # ── Phase 1: Feature Extraction (frozen base) ──────────────────
        print('  Phase 1: Feature Extraction')
        model.compile(
            optimizer=SGD(learning_rate=3e-3, momentum=0.9,
                          nesterov=True, clipnorm=1.0),
            loss=loss_fn, metrics=['accuracy'])

        model.fit(
            mixup_generator(train_gen),
            steps_per_epoch=len(train_gen),
            validation_data=val_gen,
            epochs=EPOCHS,
            callbacks=get_callbacks(),
            class_weight=class_weight_dict,
            verbose=1)

        # ── Phase 2: Fine-Tuning (unfreeze top 40 %) ──────────────────
        print('  Phase 2: Fine-Tuning')
        base.trainable = True
        freeze_up_to   = int(len(base.layers) * 0.60)
        for layer in base.layers[:freeze_up_to]:
            layer.trainable = False
        # Keep BN frozen to prevent batch-stat drift
        for layer in base.layers:
            if isinstance(layer, tf.keras.layers.BatchNormalization):
                layer.trainable = False

        model.compile(
            optimizer=SGD(learning_rate=1e-5, momentum=0.9,
                          nesterov=True, clipnorm=1.0),
            loss=loss_fn, metrics=['accuracy'])

        model.fit(
            mixup_generator(train_gen),
            steps_per_epoch=len(train_gen),
            validation_data=val_gen,
            epochs=EPOCHS,
            callbacks=get_callbacks(),
            class_weight=class_weight_dict,
            verbose=1)

        # ── Evaluate ────────────────────────────────────────────────────
        val_gen.reset()
        _, acc = model.evaluate(val_gen, verbose=0)
        print(f'  Fold {fold+1} Val Accuracy: {acc*100:.2f}%')

        if acc > best_acc:
            best_acc = acc
            # ── Serialisation-safe save ─────────────────────────────────
            # Save weights only (avoids EagerTensor JSON bug in TF 2.10)
            weights_path = f'saved_models/{model_tag}_best.weights.h5'
            model.save_weights(weights_path)
            print(f'  --> Best weights saved: {weights_path}')

        tf.keras.backend.clear_session()
        gc.collect()

    print(f'\n{model_tag} K-Fold complete. Best Val Acc: {best_acc*100:.2f}%')
    return best_acc


## Train EfficientNetB0 (K-Fold)


In [ ]:
best_acc_b0 = train_kfold(EfficientNetB0, 'efficientnetb0')



  efficientnetb0  |  Fold 1/3
Found 3306 validated image filenames belonging to 47 classes.
Found 1654 validated image filenames belonging to 47 classes.
  Phase 1: Feature Extraction
Epoch 1/80
207/207 [==============================] - 190s 803ms/step - loss: 5.9057 - accuracy: 0.0230 - val_loss: 4.6352 - val_accuracy: 0.0393 - lr: 0.0030
Epoch 2/80
207/207 [==============================] - 100s 487ms/step - loss: 5.4528 - accuracy: 0.0378 - val_loss: 4.3068 - val_accuracy: 0.1155 - lr: 0.0030
Epoch 3/80
207/207 [==============================] - 88s 428ms/step - loss: 5.1080 - accuracy: 0.0551 - val_loss: 4.0495 - val_accuracy: 0.1941 - lr: 0.0030
Epoch 4/80
207/207 [==============================] - 86s 419ms/step - loss: 4.7767 - accuracy: 0.0717 - val_loss: 3.8283 - val_accuracy: 0.2557 - lr: 0.0030
Epoch 5/80
207/207 [==============================] - 81s 395ms/step - loss: 4.5583 - accuracy: 0.1086 - val_loss: 3.6243 - val_accuracy: 0.3174 - lr: 0.0030
Epoch 6/80
207/207 [===

## Train EfficientNetB2 (K-Fold)


In [ ]:
best_acc_b2 = train_kfold(EfficientNetB2, 'efficientnetb2')


## Reload Best Models for Stacking
Because we saved **weights only**, we rebuild the architecture then load.


In [ ]:
def reload_model(base_fn, weights_path):
    model, _ = build_model(base_fn)
    model.load_weights(weights_path)
    print(f'Loaded: {weights_path}')
    return model

model_b0 = reload_model(EfficientNetB0, 'saved_models/efficientnetb0_best.weights.h5')
model_b2 = reload_model(EfficientNetB2, 'saved_models/efficientnetb2_best.weights.h5')


## Stacking — Feature Extraction


In [ ]:
def extract_softmax(model, df_subset, tag):
    """Extract softmax probabilities (no shuffle) for SVM meta-learner."""
    gen = val_datagen.flow_from_dataframe(
        df_subset, x_col='filepath', y_col='label',
        target_size=IMG_SIZE, batch_size=BATCH_SIZE,
        class_mode='categorical', shuffle=False,
        classes=classes)           # enforce consistent class order
    gen.reset()
    feats = model.predict(gen, verbose=1)
    labels = gen.classes
    print(f'{tag} features: {feats.shape}')
    return feats, labels

print('Extracting train features...')
feat_train_b0, y_train = extract_softmax(model_b0, train_val_df, 'B0-train')
feat_train_b2, _       = extract_softmax(model_b2, train_val_df, 'B2-train')

print('Extracting test features...')
feat_test_b0, y_test   = extract_softmax(model_b0, test_df, 'B0-test')
feat_test_b2, _        = extract_softmax(model_b2, test_df, 'B2-test')

X_train_meta = np.hstack([feat_train_b0, feat_train_b2])
X_test_meta  = np.hstack([feat_test_b0,  feat_test_b2])

print('Meta-train shape:', X_train_meta.shape)
print('Meta-test  shape:', X_test_meta.shape)


## SVM Meta-Learner (RBF Kernel)


In [ ]:
scaler = StandardScaler()
X_tr_scaled = scaler.fit_transform(X_train_meta)
X_te_scaled = scaler.transform(X_test_meta)

svm = SVC(kernel='rbf', C=10.0, gamma='scale',
          probability=True, random_state=42)

print('Training SVM meta-learner...')
t0 = time.time()
svm.fit(X_tr_scaled, y_train)
print(f'SVM training time: {time.time()-t0:.1f}s')

joblib.dump(svm,    'saved_models/svm_meta.joblib')
joblib.dump(scaler, 'saved_models/scaler.joblib')
print('SVM + scaler saved.')


## Final Evaluation


In [ ]:
y_pred = svm.predict(X_te_scaled)
acc    = accuracy_score(y_test, y_pred)
print(f'\n=== FINAL STACKING ENSEMBLE TEST ACCURACY: {acc*100:.2f}% ===\n')

print(classification_report(y_test, y_pred, target_names=classes))


In [ ]:
cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(22, 18))
sns.heatmap(cm, annot=False, cmap='Blues',
            xticklabels=classes, yticklabels=classes)
plt.title('Confusion Matrix — Stacking Ensemble (B0 + B2 + SVM)', fontsize=14)
plt.xlabel('Predicted', fontsize=12)
plt.ylabel('True', fontsize=12)
plt.xticks(rotation=90, fontsize=7)
plt.yticks(rotation=0,  fontsize=7)
plt.tight_layout()
plt.savefig('saved_models/confusion_matrix.png', dpi=150)
plt.show()
print('Confusion matrix saved.')


In [ ]:
print('\nPer-class accuracy:')
for i, cls in enumerate(classes):
    idx = np.where(y_test == i)[0]
    if len(idx):
        a = np.mean(y_pred[idx] == i)
        print(f'  {cls:40s}: {a*100:5.1f}%  ({len(idx)} samples)')
